In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']

    # ---- Temperature ----
    data['temp_range'] = (
        data['MaxOfUpperTRange'] -
        data['MinOfLowerTRange']
    )

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']

    # ---- Non-linear ----
    for col in ['clonesize', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Single Clustering (🔥 فقط یکی) ----
    cluster_cols = ['clonesize', 'avg_temp', 'RainingDays', 'fruitmass']

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=7,          # 🔥 sweet spot
            random_state=SEED,
            n_init=40
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    return data, kmeans, scaler


train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

In [4]:
X = train_fe.drop(columns=['yield'])
y = train_fe['yield']

In [5]:
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

In [6]:
# =========================================================
# =======================
# STAGE 1 — MAIN MODEL
# =======================
params_stage1 = {
    'objective': 'regression_l1',
    'metric': 'mae',

    'learning_rate': 0.022,
    'num_leaves': 72,
    'min_data_in_leaf': 45,

    'feature_fraction': 0.88,
    'bagging_fraction': 0.88,
    'bagging_freq': 1,

    'lambda_l1': 0.6,
    'lambda_l2': 1.2,

    'boosting': 'gbdt',
    'max_depth': -1,
    'verbosity': -1,
    'seed': SEED,

    'device': 'gpu'
}

oof_stage1 = np.zeros(len(X))
test_stage1 = np.zeros(len(test_fe))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'[Stage 1] Fold {fold}/10')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, y_tr)
    val_set   = lgb.Dataset(X_val, y_val)

    model = lgb.train(
        params_stage1,
        train_set,
        num_boost_round=5200,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(300, verbose=False)]
    )

    oof_stage1[val_idx] = model.predict(X_val)
    test_stage1 += model.predict(test_fe) / kf.n_splits

print('\nStage-1 MAE:',
      mean_absolute_error(y, oof_stage1))

[Stage 1] Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[Stage 1] Fold 2/10
[Stage 1] Fold 3/10
[Stage 1] Fold 4/10
[Stage 1] Fold 5/10
[Stage 1] Fold 6/10
[Stage 1] Fold 7/10
[Stage 1] Fold 8/10
[Stage 1] Fold 9/10
[Stage 1] Fold 10/10

Stage-1 MAE: 245.37593575248016


In [7]:
# =======================
# STAGE 2 — RESIDUAL MODEL (KILLER)
# =======================
residual = y - oof_stage1

params_stage2 = {
    'objective': 'regression_l2',
    'metric': 'rmse',

    'learning_rate': 0.03,
    'num_leaves': 32,
    'min_data_in_leaf': 80,

    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 1,

    'lambda_l1': 0.0,
    'lambda_l2': 2.0,

    'verbosity': -1,
    'seed': SEED,

    'device': 'gpu'
}

oof_stage2 = np.zeros(len(X))
test_stage2 = np.zeros(len(test_fe))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'[Stage 2] Fold {fold}/10')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    r_tr, r_val = residual.iloc[tr_idx], residual.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, r_tr)
    val_set   = lgb.Dataset(X_val, r_val)

    model_res = lgb.train(
        params_stage2,
        train_set,
        num_boost_round=2500,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )

    oof_stage2[val_idx] = model_res.predict(X_val)
    test_stage2 += model_res.predict(test_fe) / kf.n_splits

# =========================================================
# FINAL PREDICTION
# =========================================================
oof_final = oof_stage1 + oof_stage2
test_final = test_stage1 + test_stage2

final_mae = mean_absolute_error(y, oof_final)
print('\n🔥🔥 FINAL OOF MAE:', final_mae)


[Stage 2] Fold 1/10
[Stage 2] Fold 2/10
[Stage 2] Fold 3/10
[Stage 2] Fold 4/10
[Stage 2] Fold 5/10
[Stage 2] Fold 6/10
[Stage 2] Fold 7/10
[Stage 2] Fold 8/10
[Stage 2] Fold 9/10
[Stage 2] Fold 10/10

🔥🔥 FINAL OOF MAE: 247.42299048263226


In [8]:
test_final = np.clip(
    test_final,
    y.min(),
    y.max()
)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': test_final
})

submission.to_csv('submission.csv', index=False)
submission.head()


,id,yield
0,15000,7497.533572
1,15001,5896.206556
2,15002,6595.055522
3,15003,4693.133428
4,15004,5913.217230
